# Pydantic: turn uncertain input into an explicit ML data contract

A delivery-estimate service receives JSON from websites and batch uploads. Some records contain numeric strings, impossible weights, misspelled fields, unknown service types or deadlines before the request was made. If those values reach feature engineering silently, a plausible prediction can hide an invalid input.

We will define the contract, inspect validation behavior, audit a mixed batch, build features only from accepted requests and export a JSON Schema. The final estimator is an explicitly labeled teaching rule, not a trained logistics model. The focus is validation and traceability rather than forecasting accuracy.

All data, explanations, examples and exercises are inside this notebook. It runs offline after installing dependencies, requires no server, and uses Pydantic v2 APIs. Before each validation, predict whether the record should be accepted, converted or rejected—and why.

## 1. Keep validation separate from learning

Pydantic validates and serializes structured Python/JSON data. It is not a DataFrame engine, statistical quality monitor or predictive model. A valid record can still be inaccurate, stale or out of distribution.

Use the repository's `pydantic==2.13.3`, pandas and numpy. The examples use `model_validate`, `field_validator`, `model_validator` and `model_dump`, rather than older v1 method names. Generated audit reports, feature tables and schema files live under `outputs/pydantic/` and are replaced on rerun. These are project outputs, not separate teaching guides.

In [ ]:
from pathlib import Path
from typing import Annotated, Literal, Optional, Union
from datetime import datetime, timezone
import copy, json, platform
import numpy as np
import pandas as pd
import pydantic
from pydantic import (BaseModel, ConfigDict, Field, ValidationError, field_validator,
                      model_validator, computed_field, TypeAdapter, AwareDatetime,
                      FiniteFloat, SecretStr)
from IPython.display import display

OUT=Path('outputs/pydantic');OUT.mkdir(parents=True,exist_ok=True)
print({'python':platform.python_version(),'pydantic':pydantic.__version__})

def show_errors(exc):
    # Keep error location/type/message; avoid echoing arbitrary request bodies.
    return pd.DataFrame(exc.errors(include_input=False,include_context=False,include_url=False))

## 2. Decide whether a string is an acceptable number

In ordinary mode Pydantic can convert compatible input, such as a numeric string into an integer. That can be useful at an ingestion boundary, but it is a policy choice rather than proof that the source data was well-typed.

Compare the same payload under ordinary and strict integer validation. Also test a Boolean: Python treats `bool` as related to `int`, but strict integer validation should reject it. Use strictness selectively when the transport representation needs parsing, such as timestamps arriving as JSON strings.

Strict mode is type- and input-format-specific. JSON cannot natively represent every Python type; strict JSON handling of dates differs from validating a Python string as a date. Do not summarize strictness as “nothing is ever converted.”

In [ ]:
class ParsedCount(BaseModel):
    count:int
class ExactCount(BaseModel):
    count:Annotated[int,Field(strict=True,ge=1)]

parsed=ParsedCount.model_validate({'count':'3'})
assert parsed.count==3 and isinstance(parsed.count,int)
print('Ordinary parsing:',parsed)
for value in ['3',True,0]:
    try:
        ExactCount.model_validate({'count':value})
    except ValidationError as exc:
        display(show_errors(exc))
    else:
        raise AssertionError(f'Expected rejection for {value!r}')
assert ExactCount(count=3).count==3

## 3. Specify the unit and range of each physical quantity

A weight of 2.5 is ambiguous without a unit. The schema names kilograms and centimeters and uses finite numeric values. We reject NaN/infinity, nonpositive dimensions and weights above the chosen teaching limit. These limits are fictional service rules, not carrier regulations.

A parcel has a description, weight and exactly three dimensions. Descriptions are trimmed, while numeric strings remain disallowed. Pydantic parses JSON arrays into tuples for dimensions; that structural conversion is intentional.

The base model forbids extra fields so a typo such as `weigth_kg` is visible. It validates defaults and direct field assignment. That does not imply that every mutation inside a nested Python list will be intercepted.

In [ ]:
Weight=Annotated[FiniteFloat,Field(strict=True,gt=0,le=70)]
Dimension=Annotated[FiniteFloat,Field(strict=True,gt=0,le=300)]

class Contract(BaseModel):
    model_config=ConfigDict(extra='forbid',validate_assignment=True,validate_default=True)

class Parcel(Contract):
    description:Annotated[str,Field(min_length=1,max_length=120)]
    weight_kg:Weight
    dimensions_cm:tuple[Dimension,Dimension,Dimension]

    @field_validator('description',mode='before')
    @classmethod
    def trim_description(cls,value):
        return value.strip() if isinstance(value,str) else value

parcel=Parcel.model_validate({'description':'  Books  ','weight_kg':2.5,'dimensions_cm':[30.0,20.0,10.0]})
assert parcel.description=='Books' and parcel.dimensions_cm==(30.0,20.0,10.0)
print(parcel)
for invalid in ['2.5',float('nan'),float('inf'),True,-1.0]:
    try:
        Parcel.model_validate({'description':'Books','weight_kg':invalid,'dimensions_cm':[30.0,20.0,10.0]})
    except ValidationError as exc:
        print(repr(invalid),exc.errors()[0]['type'])
    else:
        raise AssertionError('Invalid weight was accepted')

## 4. Let an explicit discriminator select a service contract

A standard shipment specifies dispatch business days; express specifies a cutoff hour. A discriminated union uses `kind` to select the right model and then validates that model's required fields.

Without a discriminator, several superficially similar schemas can create confusing branch errors or unintended matches. Here an unknown service kind is an error, not a reason to guess standard delivery. Nested models retain their own extra-field policy.

We also define source metadata and tags. `default_factory=list` makes each context's tag list independent. An optional field with `=None` may be omitted; `Optional[T]` without a default in Pydantic v2 permits null but still requires the field to be present.

In [ ]:
class StandardService(Contract):
    kind:Literal['standard']
    dispatch_business_days:Annotated[int,Field(strict=True,ge=1,le=5)]=2
class ExpressService(Contract):
    kind:Literal['express']
    cutoff_hour:Annotated[int,Field(strict=True,ge=0,le=23)]=16
Service=Annotated[Union[StandardService,ExpressService],Field(discriminator='kind')]

class SourceContext(Contract):
    source:Literal['web','batch']
    customer_note:Optional[str]=None
    tags:list[str]=Field(default_factory=list,max_length=10)

first=SourceContext(source='web');second=SourceContext(source='web')
first.tags.append('demo')
assert second.tags==[]
service_adapter=TypeAdapter(Service)
print(service_adapter.validate_python({'kind':'express','cutoff_hour':15}))
try:
    service_adapter.validate_python({'kind':'overnight'})
except ValidationError as exc:
    display(show_errors(exc))

## 5. Check relationships that no single field can establish

A request includes a normalized identifier, an aware request timestamp, positive distance, one to five parcels, service choice and source metadata. A deadline may be omitted, but when provided it must follow the request timestamp. Total parcel weight cannot exceed 200 kg under this fictional contract.

A field validator normalizes identifiers before their pattern check. It only transforms strings; it does not stringify arbitrary objects. A model validator checks relationships after individual fields have been parsed. Raise `ValueError` for input violations so Pydantic includes them in a structured validation error; do not use assertions for production input checks.

`computed_field` exposes derived total weight during serialization. It does not become an accepted input field, and it is not a learned feature transformation. Timestamps must include a timezone; this prevents silently comparing naive local times with UTC.

In [ ]:
class DeliveryRequest(Contract):
    schema_version:Literal['1']='1'
    request_id:Annotated[str,Field(pattern=r'^REQ-\d{3}$')]
    requested_at:AwareDatetime
    delivery_deadline:Optional[AwareDatetime]=None
    distance_km:Annotated[FiniteFloat,Field(strict=True,gt=0,le=20000)]
    parcels:list[Parcel]=Field(min_length=1,max_length=5)
    service:Service
    context:SourceContext

    @field_validator('request_id',mode='before')
    @classmethod
    def normalize_id(cls,value):
        return value.strip().upper() if isinstance(value,str) else value

    @model_validator(mode='after')
    def check_request_relationships(self):
        if self.delivery_deadline is not None and self.delivery_deadline<=self.requested_at:
            raise ValueError('delivery_deadline must be after requested_at')
        if sum(p.weight_kg for p in self.parcels)>200:
            raise ValueError('total parcel weight must not exceed 200 kg')
        return self

    @computed_field
    @property
    def total_weight_kg(self)->float:
        return sum(p.weight_kg for p in self.parcels)

base={'schema_version':'1','request_id':' req-001 ','requested_at':'2026-01-10T09:00:00Z',
      'distance_km':180.0,'parcels':[{'description':'Books','weight_kg':2.5,'dimensions_cm':[30.0,20.0,10.0]}],
      'service':{'kind':'standard','dispatch_business_days':2},'context':{'source':'web'}}
validated=DeliveryRequest.model_validate(base)
assert validated.request_id=='REQ-001' and validated.total_weight_kg==2.5
print(validated)

## 6. Make rejection paths part of the lesson

Predict where each invalid example should fail: a nested numeric field, missing timezone, service discriminator, or cross-field rule. Error locations are paths into the payload. A model-wide error can have an empty location because no single field owns the violated relationship.

The example below verifies several independent boundaries. These checks test contract behavior, not the validity of any real shipment. A JSON request within all limits can still contain false measurements.

In [ ]:
invalid_cases={}
x=copy.deepcopy(base);x['requested_at']='2026-01-10T09:00:00';invalid_cases['missing timezone']=x
x=copy.deepcopy(base);x['delivery_deadline']='2026-01-09T09:00:00Z';invalid_cases['deadline before request']=x
x=copy.deepcopy(base);x['parcels'][0]['dimensions_cm']=[30.0,20.0];invalid_cases['missing dimension']=x
x=copy.deepcopy(base);x['service']={'kind':'express','cutoff_hour':True};invalid_cases['Boolean cutoff']=x
x=copy.deepcopy(base);x['parcels']=[{'description':'Heavy','weight_kg':70.0,'dimensions_cm':[10.0,10.0,10.0]} for _ in range(3)];invalid_cases['total mass']=x
for name,payload in invalid_cases.items():
    try:
        DeliveryRequest.model_validate(payload)
    except ValidationError as exc:
        print(name);display(show_errors(exc))
    else:
        raise AssertionError(f'Expected rejection: {name}')

## 7. Build a batch whose quality can be reconciled

The twelve-record fixture has six valid requests and six deliberately invalid ones. Valid examples vary distance, service, parcel count and deadline. Invalid examples contain negative distance, string weight, no parcels, an early deadline, a misspelled extra field and an unknown service kind.

Preserve row numbers and declared request IDs in an audit. Do not coerce every invalid value merely to increase acceptance. The expected count is six accepted plus six rejected, with no source row lost between them.

This is synthetic input authored for the exercise. Its failure rates are not estimates of a real service's data quality.

In [ ]:
raw=[]
for i in range(1,13):
    row=copy.deepcopy(base);row['request_id']=f'REQ-{i:03d}';raw.append(row)
raw[1]['distance_km']=720.0
raw[2]['service']={'kind':'express','cutoff_hour':15}
raw[3]['parcels'].append({'description':'Clothes','weight_kg':1.0,'dimensions_cm':[20.0,15.0,10.0]})
raw[4]['delivery_deadline']='2026-01-15T09:00:00Z'
raw[5]['context']={'source':'batch','customer_note':None,'tags':['imported']}
raw[6]['distance_km']=-5.0
raw[7]['parcels'][0]['weight_kg']='2.5'
raw[8]['parcels']=[]
raw[9]['delivery_deadline']='2026-01-09T09:00:00Z'
raw[10]['distnace_km']=100.0
raw[11]['service']={'kind':'overnight'}
assert len(raw)==12
(OUT/'raw_requests.json').write_text(json.dumps(raw,indent=2))

## 8. Quarantine invalid rows without hiding their reasons

For a partial-ingestion policy, validate each row and collect all reported errors for that row. This keeps valid records usable and invalid records reviewable. Some applications require an atomic batch instead; the policy should be explicit.

Error types support grouping, but a row can have multiple errors, so an error-count total may exceed the number of rejected rows. Avoid logging arbitrary raw inputs by default. The fixture is harmless, but real requests can contain sensitive customer notes.

In [ ]:
accepted=[];rejected=[]
for row_number,payload in enumerate(raw):
    try:
        request=DeliveryRequest.model_validate(payload)
    except ValidationError as exc:
        rejected.append({'row_number':row_number,'request_id':payload.get('request_id'),
                         'errors':exc.errors(include_input=False,include_context=False,include_url=False)})
    else:
        accepted.append({'row_number':row_number,'request':request})
assert len(accepted)==6 and len(rejected)==6
accepted_rows={r['row_number'] for r in accepted};rejected_rows={r['row_number'] for r in rejected}
assert accepted_rows.isdisjoint(rejected_rows)
assert accepted_rows|rejected_rows==set(range(len(raw)))
error_rows=[{'row_number':row['row_number'],'request_id':row['request_id'],
             'path':'.'.join(map(str,error['loc'])),'type':error['type'],'message':error['msg']}
            for row in rejected for error in row['errors']]
audit=pd.DataFrame(error_rows)
display(audit)
display(audit.groupby('type').size().rename('error_count'))

## 9. Distinguish row validity from dataset-level integrity

`TypeAdapter` validates a container without introducing a wrapper model. Validating the full mixed list reports batch errors; it does not return a partial list of successes. Reuse an adapter for repeated validations rather than rebuilding its schema per row.

Two individually valid requests can still have the same ID. Uniqueness across an upload, foreign-key existence and idempotency across earlier uploads require batch/application/storage checks beyond the row schema. We demonstrate an in-batch duplicate check separately.

In [ ]:
batch_adapter=TypeAdapter(list[DeliveryRequest])
try:
    batch_adapter.validate_python(raw)
except ValidationError as exc:
    print('Whole-list validation reports',exc.error_count(),'errors; it returns no partially accepted list.')
else:
    raise AssertionError('Mixed batch should not validate as a whole')

valid_payloads=[item['request'].model_dump(mode='json',exclude_computed_fields=True) for item in accepted]
whole_valid_batch=batch_adapter.validate_python(valid_payloads)
assert len(whole_valid_batch)==6

def require_unique_ids(requests):
    ids=[request.request_id for request in requests]
    if len(ids)!=len(set(ids)):
        raise ValueError('Duplicate request_id within batch')
require_unique_ids(whole_valid_batch)
try:
    require_unique_ids(whole_valid_batch+[whole_valid_batch[0]])
except ValueError as exc:
    print('Expected dataset-level check:',exc)

## 10. Derive features only after the contract is satisfied

Use named columns with explicit units: distance, mass, volume, parcel count and express flag. A validated deadline is not used as a feature in this teaching estimator: a customer's desired outcome does not necessarily describe transport time, and feature availability must match the decision moment.

For demonstration, estimated hours equal dispatch allowance plus distance/60 plus half an hour per parcel. Standard dispatch business days are crudely mapped to 24-hour periods; weekends, calendars, routing, weather, cutoffs and carrier capacity are omitted. The express cutoff field is validated but not modeled. **This is a toy rule, not a real ETA model or delivery promise.**

Predict which accepted request changes the estimate and why. Validate output finiteness, but recognize that a well-typed estimate can still be operationally wrong.

In [ ]:
class Estimate(Contract):
    request_id:str
    estimated_hours:Annotated[FiniteFloat,Field(gt=0)]
    estimator:Literal['teaching-rule-v1']='teaching-rule-v1'

features=[];estimates=[]
for item in accepted:
    request=item['request']
    volume=sum(np.prod(parcel.dimensions_cm) for parcel in request.parcels)
    is_express=isinstance(request.service,ExpressService)
    features.append({'request_id':request.request_id,'distance_km':request.distance_km,
                     'weight_kg':request.total_weight_kg,'volume_cm3':float(volume),
                     'parcel_count':len(request.parcels),'is_express':int(is_express)})
    dispatch=4.0 if is_express else 24.0*request.service.dispatch_business_days
    hours=dispatch+request.distance_km/60.0+0.5*len(request.parcels)
    estimates.append(Estimate(request_id=request.request_id,estimated_hours=hours))
features=pd.DataFrame(features)
assert len(features)==len(accepted) and features.request_id.is_unique
assert np.isfinite(features.drop(columns='request_id').to_numpy()).all()
display(features)
display(pd.DataFrame([e.model_dump() for e in estimates]))

## 11. Serialize with the receiving system in mind

`model_dump()` returns Python values, which can include datetime objects and tuples. `mode='json'` returns JSON-compatible values; `model_dump_json()` directly returns a JSON string. Computed fields appear in output by default, so exclude them for a strict round trip into an input schema that forbids extras.

Aliases support external field names while preserving clearer Python attribute names. Use `by_alias=True` when the external contract expects them. `exclude_unset` and `exclude_none` answer different questions: a field deliberately set to null is not the same as an omitted field.

Do not remove nulls or defaults automatically if the receiving system distinguishes them.

In [ ]:
request=accepted[0]['request']
python_payload=request.model_dump(exclude_computed_fields=True)
assert isinstance(python_payload['requested_at'],datetime)
json_text=request.model_dump_json(exclude_computed_fields=True)
roundtrip=DeliveryRequest.model_validate_json(json_text)
assert roundtrip==request
print('JSON timestamp:',json.loads(json_text)['requested_at'])

class PartnerRecord(BaseModel):
    model_config=ConfigDict(populate_by_name=True,extra='forbid')
    request_id:str=Field(alias='requestId')
partner=PartnerRecord.model_validate({'requestId':'REQ-001'})
assert partner.request_id=='REQ-001'
print(partner.model_dump(by_alias=True))
omitted=SourceContext(source='web')
explicit=SourceContext(source='web',customer_note=None)
print('Omitted:',omitted.model_dump(exclude_unset=True))
print('Explicit null:',explicit.model_dump(exclude_unset=True))

## 12. Learn where validation stops

`validate_assignment=True` checks direct attribute assignment. It does not observe every in-place mutation inside a list or dictionary. `model_copy(update=...)` trusts its update values instead of validating them, and `model_construct` deliberately bypasses validation. Neither should be an untrusted-input path.

To apply a user-supplied patch, merge it into a JSON-compatible dump and validate a new model. This rechecks cross-field rules without relying on in-place mutation. Frozen models restrict assignment, but nested mutable containers still need careful design.

SecretStr masks normal display; it is not encryption or access control, and the underlying value can still be retrieved explicitly. Keep actual credentials outside teaching examples and exported records.

In [ ]:
count=ExactCount(count=3)
unchecked=count.model_copy(update={'count':-1})
assert unchecked.count==-1
try:
    ExactCount.model_validate({**count.model_dump(),'count':-1})
except ValidationError:
    print('Revalidation caught the invalid patch that model_copy trusted.')
else:
    raise AssertionError('Negative count should fail')

safe_patch={**request.model_dump(mode='json',exclude_computed_fields=True),'distance_km':240.0}
updated=DeliveryRequest.model_validate(safe_patch)
assert updated.distance_km==240.0 and request.distance_km==180.0
try:
    updated.distance_km=-1.0
except ValidationError:
    print('Direct invalid field assignment rejected.')
else:
    raise AssertionError('Assignment validation should reject negative distance')

class DemoCredentials(BaseModel):
    token:SecretStr
credentials=DemoCredentials(token='fictional-token')
assert 'fictional-token' not in repr(credentials)
print(credentials)

## 13. Export a schema and explain what it cannot express

JSON Schema describes field types, bounds, required properties and union discriminators. It helps clients generate forms or validate basic structure. Arbitrary Python validators, such as comparing two dates or adding all parcel weights, are not automatically translated into equivalent JSON Schema logic.

Export both the validation schema and audit artifacts. The schema is an interface document, not proof of data truth, a learned model or a complete description of all business rules. Keep a schema version and migration policy when changing accepted formats; silent coercion is not a versioning strategy.

For a real API, FastAPI can consume these models directly. For large tabular datasets, vectorized DataFrame validation may be more appropriate than constructing a Python object for every cell. Choose the boundary you need to validate.

In [ ]:
schema=DeliveryRequest.model_json_schema(mode='validation')
assert schema['additionalProperties'] is False
assert 'request_id' in schema['required'] and 'delivery_deadline' not in schema['required']
assert 'total_weight_kg' not in schema['properties']
assert schema['properties']['service']['discriminator']['propertyName']=='kind'
(OUT/'delivery_request.schema.json').write_text(json.dumps(schema,indent=2))
(OUT/'accepted_requests.json').write_text(json.dumps(valid_payloads,indent=2))
(OUT/'rejected_requests.json').write_text(json.dumps(rejected,indent=2))
(OUT/'estimates.json').write_text(json.dumps([e.model_dump(mode='json') for e in estimates],indent=2))
audit.to_csv(OUT/'validation_audit.csv',index=False)
features.to_csv(OUT/'accepted_features.csv',index=False)
reloaded=batch_adapter.validate_json((OUT/'accepted_requests.json').read_text())
assert [r.request_id for r in reloaded]==[r.request_id for r in whole_valid_batch]
print({'source_rows':len(raw),'accepted':len(reloaded),'rejected':len(rejected),'feature_rows':len(features)})

## 14. Choose the appropriate validation entry point

| Need | API / feature | Decision to make |
|---|---|---|
| Validate a Python dictionary | `model_validate` | Coercion policy and extras |
| Validate a JSON body | `model_validate_json` | JSON representation and required fields |
| Validate a list or union | `TypeAdapter` | Atomic batch versus per-row handling |
| Normalize one field | `field_validator(mode='before')` | Which transformations are justified? |
| Check related fields | `model_validator(mode='after')` | Which combinations are invalid? |
| Derive output-only data | `computed_field` | Should it be present in the external response? |
| Export a stable interface | `model_json_schema` | Which Python rules need extra documentation? |
| Read application settings | Separate `pydantic-settings` package | Sources, precedence and secret handling |

Pydantic dataclasses can validate dataclass-style objects; `validate_call` can validate function arguments. Use them when they clarify a boundary, not as substitutes for designing one. Custom serializers change output representation and deserve round-trip or consumer-contract checks.

Structured LLM output can also be validated against a model. A parsed field of the correct type can still be a hallucinated fact; schema validation does not establish truth or authorization.

## 15. Explain what the evidence supports

1. Add a record with two independent invalid fields. Verify that the rejected-row count increases by one while error counts may increase by two.
2. Decide whether numeric strings should be accepted from a CSV import but rejected from an API. Implement separate boundary policies without weakening the shared physical constraints.
3. Add a duplicate ID from an earlier upload. Explain why in-batch uniqueness alone is insufficient and where idempotency should live.
4. Add a timezone-offset example whose deadline is later in local display but earlier in UTC. Predict the relationship check before running it.
5. Replace the teaching rule with a trained estimator. Specify feature availability, units, model version and output schema separately from prediction quality evaluation.

| Observation | Investigate |
|---|---|
| A number string was accepted | Default coercion versus field strictness |
| A typo vanished | Extra-field policy |
| JSON output cannot be revalidated | Computed fields, aliases and schema differences |
| Direct assignment is checked but list edits are not | In-place mutation boundary |
| A valid-looking record duplicates an earlier one | Dataset/storage integrity rather than row validation |
| Schema consumers accept a bad date relationship | Python model validators not encoded by JSON Schema |

Official references: [models](https://docs.pydantic.dev/latest/concepts/models/), [validators](https://docs.pydantic.dev/latest/concepts/validators/), [strict mode](https://docs.pydantic.dev/latest/concepts/strict_mode/), [serialization](https://docs.pydantic.dev/latest/concepts/serialization/), [unions](https://docs.pydantic.dev/latest/concepts/unions/), and [TypeAdapter](https://docs.pydantic.dev/latest/concepts/type_adapter/). The notebook prints the exact installed v2 version.

## Second project: migrate a mixed-version feature feed without silently trusting updates

A scoring feed contains legacy numeric strings, a newer strict schema, unknown fields and conflicting IDs. Design explicit version adapters, distinguish normalization from acceptance, and compare per-row quarantine with whole-batch validation. This generated fixture models a schema migration, not a live customer's data.

Run these cells in order. This extension uses its own `p2_` variables and writes only its named outputs under `outputs/project2/`. Predict each comparison before running it; the worked discussion after the code explains the result and its limits.

### Work through an unsafe patch

`model_copy(update=...)` assumes the update is trusted; it does not rerun field validators. A valid feature object copied with a negative age can therefore become invalid. Merge an untrusted patch into a dictionary and validate a new object instead. The original remains intact if validation fails.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import time
from IPython.display import display
p2_out=Path('outputs/project2');p2_out.mkdir(parents=True,exist_ok=True)

from pydantic import BaseModel,ConfigDict,Field,ValidationError,TypeAdapter,model_validator
from typing import Annotated,Literal
class P2Features(BaseModel):
    model_config=ConfigDict(extra='forbid')
    version:Literal['2']='2'
    entity_id:Annotated[int,Field(strict=True,gt=0)]
    age_days:Annotated[int,Field(strict=True,ge=0,le=36500)]
    purchases:Annotated[int,Field(strict=True,ge=0)]
    spend_cents:Annotated[int,Field(strict=True,ge=0)]
    @model_validator(mode='after')
    def consistent(self):
        if self.purchases==0 and self.spend_cents!=0:raise ValueError('spend without purchases')
        return self
p2_original=P2Features(entity_id=1,age_days=10,purchases=1,spend_cents=100)
assert p2_original.model_copy(update={'age_days':-1}).age_days==-1
try:P2Features.model_validate({**p2_original.model_dump(),'age_days':-1})
except ValidationError:print('Untrusted patch rejected through revalidation.')
else:raise AssertionError('Negative age accepted')

### Work through explicit migration and validation budgets

Only version 1 permits numeric strings, and only a canonical digits-only representation is converted. Version 2 never receives this convenience conversion. Unknown versions fail. The adapter handles representation; Pydantic still checks range and relationships afterward.

An atomic list adapter is appropriate when every row must pass together. Row-by-row handling supports partial ingestion and a rejection audit. Reuse compiled adapters/models, bound batch sizes, and time equivalent valid populations rather than calling partial and atomic policies interchangeable.

In [ ]:
def p2_migrate(row):
    row=dict(row)
    if row.get('version')=='1':
        if set(row)!={'version','entity_id','age_days','purchases','spend_cents'}:raise ValueError('legacy field mismatch')
        for key in ['entity_id','age_days','purchases','spend_cents']:
            if not isinstance(row[key],str) or not row[key].isascii() or not row[key].isdigit():raise ValueError('noncanonical legacy number')
            row[key]=int(row[key])
        row['version']='2'
    return P2Features.model_validate(row)
p2_feed=[{'version':'1','entity_id':'1','age_days':'10','purchases':'1','spend_cents':'100'},
 {'version':'2','entity_id':2,'age_days':20,'purchases':0,'spend_cents':0},
 {'version':'2','entity_id':3,'age_days':'30','purchases':1,'spend_cents':100},
 {'version':'2','entity_id':4,'age_days':30,'purchases':0,'spend_cents':100},
 {'version':'3','entity_id':5,'age_days':30,'purchases':1,'spend_cents':100},
 {'version':'2','entity_id':1,'age_days':99,'purchases':1,'spend_cents':100}]
p2_good={};p2_bad=[]
for p2_i,p2_row in enumerate(p2_feed):
    try:
        p2_value=p2_migrate(p2_row)
        if p2_value.entity_id in p2_good and p2_good[p2_value.entity_id]!=p2_value:raise ValueError('conflicting ID')
        p2_good[p2_value.entity_id]=p2_value
    except (ValidationError,ValueError) as p2_error:p2_bad.append({'row':p2_i,'reason':str(p2_error).splitlines()[0]})
assert len(p2_good)==2 and len(p2_bad)==4
p2_adapter=TypeAdapter(list[P2Features])
p2_valid_rows=[v.model_dump() for v in p2_good.values()]*1000
p2_t=time.perf_counter();p2_atomic=p2_adapter.validate_python(p2_valid_rows);p2_atomic_time=time.perf_counter()-p2_t
p2_t=time.perf_counter();p2_individual=[P2Features.model_validate(r) for r in p2_valid_rows];p2_row_time=time.perf_counter()-p2_t
assert p2_atomic==p2_individual
pd.DataFrame(p2_bad).to_csv(p2_out/'pydantic_migration_rejections.csv',index=False)
display(pd.DataFrame(p2_bad));print({'valid_rows_timed':2000,'list_seconds':p2_atomic_time,'row_seconds':p2_row_time})

### Explain why stricter is not always the entire policy

The legacy adapter deliberately converts an approved representation; the new schema remains strict. Four rows are rejected for distinct reasons, including a cross-record conflict that no individual model could catch. Timing 2,000 valid objects isolates call overhead; it does not compare the behavior of a mixed invalid batch.

**Try:** send version 1's age as `"10.0"`. **Worked reasoning:** reject it under the declared canonical integer policy, even though Python could parse it through a float. If compatibility requires that representation, change the version adapter deliberately and test edge cases such as exponent notation and fractional values before widening acceptance.